# SaaS Customer Churn — Feature Engineering

**Goal:** Convert EDA findings into model-ready features. Every feature
built here is grounded in a pattern we observed in notebook 01.

**Input:** Raw Telco dataset (7,043 rows, 21 columns)  
**Output:** Cleaned, engineered feature matrix ready for modeling

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('muted')

df = pd.read_csv('https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv')

# Fixes from notebook 01
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
df['Churn_Binary'] = (df['Churn'] == 'Yes').astype(int)

print(f"Shape: {df.shape}")
print("Data loaded and cleaned.")

Shape: (7043, 22)
Data loaded and cleaned.


## 1. Tenure features

From EDA: churn drops sharply after 12 months and again after 24 months.
We encode this as both a binned group and a continuous risk score.

In [ ]:
# Tenure bucket
df['Tenure_Group'] = pd.cut(df['tenure'],
                             bins=[0, 12, 24, 48, 72],
                             labels=['0-12 months', '13-24 months',
                                     '25-48 months', '49-72 months'])

# Early tenure flag — highest risk window
df['Is_New_Customer'] = (df['tenure'] <= 12).astype(int)

print("Tenure features created.")
print(f"\nNew customers (tenure <= 12): {df['Is_New_Customer'].sum():,} ({df['Is_New_Customer'].mean():.1%})")
print(f"\nTenure group distribution:\n{df['Tenure_Group'].value_counts().sort_index()}")

Tenure features created.

New customers (tenure <= 12): 2,186 (31.0%)

Tenure group distribution:
Tenure_Group
0-12 months     2175
13-24 months    1024
25-48 months    1594
49-72 months    2239
Name: count, dtype: int64


## 2. Contract and payment features

From EDA: month-to-month contracts churn at 42.7% and electronic
check customers at 45.3%. Both get binary risk flags.

In [ ]:
# High risk contract flag
df['Is_Monthly_Contract'] = (df['Contract'] == 'Month-to-month').astype(int)

# Electronic check flag
df['Is_Electronic_Check'] = (df['PaymentMethod'] == 'Electronic check').astype(int)

# Paperless billing flag — correlated with digital-first, higher churn
df['Is_Paperless'] = (df['PaperlessBilling'] == 'Yes').astype(int)

print("Contract and payment features created.")
print(f"\nMonth-to-month customers: {df['Is_Monthly_Contract'].sum():,} ({df['Is_Monthly_Contract'].mean():.1%})")
print(f"Electronic check customers: {df['Is_Electronic_Check'].sum():,} ({df['Is_Electronic_Check'].mean():.1%})")
print(f"Paperless billing customers: {df['Is_Paperless'].sum():,} ({df['Is_Paperless'].mean():.1%})")

Contract and payment features created.

Month-to-month customers: 3,875 (55.0%)
Electronic check customers: 2,365 (33.6%)
Paperless billing customers: 4,171 (59.2%)


## 3. Service features

From EDA: customers without tech support or online security churn
at nearly 3x the rate. We create individual flags and a combined
protective services score.

In [ ]:
# Individual protective service flags
df['Has_TechSupport'] = (df['TechSupport'] == 'Yes').astype(int)
df['Has_OnlineSecurity'] = (df['OnlineSecurity'] == 'Yes').astype(int)
df['Has_OnlineBackup'] = (df['OnlineBackup'] == 'Yes').astype(int)
df['Has_DeviceProtection'] = (df['DeviceProtection'] == 'Yes').astype(int)

# Protective services score — count of protective services subscribed
df['Protective_Services_Score'] = (
    df['Has_TechSupport'] +
    df['Has_OnlineSecurity'] +
    df['Has_OnlineBackup'] +
    df['Has_DeviceProtection']
)

# High risk flag — no protective services at all
df['No_Protective_Services'] = (df['Protective_Services_Score'] == 0).astype(int)

print("Service features created.")
print(f"\nProtective services score distribution:")
print(df['Protective_Services_Score'].value_counts().sort_index())
print(f"\nCustomers with zero protective services: {df['No_Protective_Services'].sum():,} ({df['No_Protective_Services'].mean():.1%})")

Service features created.

Protective services score distribution:
Protective_Services_Score
0    2793
1    1467
2    1372
3     941
4     470
Name: count, dtype: int64

Customers with zero protective services: 2,793 (39.7%)


## 4. Charge features

Higher monthly charges combined with a month-to-month contract is
the most dangerous combination. We create a charge per service metric
and a high-charge flag.

In [ ]:
# Total number of services subscribed
service_cols = ['PhoneService', 'MultipleLines', 'InternetService',
                'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                'TechSupport', 'StreamingTV', 'StreamingMovies']

df['Total_Services'] = (df[service_cols] == 'Yes').sum(axis=1)

# Charge per service — normalizes monthly charge by how much they use
df['Charge_Per_Service'] = df['MonthlyCharges'] / (df['Total_Services'] + 1)

# High monthly charge flag — above 75th percentile
charge_75 = df['MonthlyCharges'].quantile(0.75)
df['Is_High_Charge'] = (df['MonthlyCharges'] > charge_75).astype(int)

# Most dangerous combination — month-to-month AND high charge
df['High_Risk_Combo'] = (
    (df['Is_Monthly_Contract'] == 1) &
    (df['Is_High_Charge'] == 1)
).astype(int)

print("Charge features created.")
print(f"\n75th percentile monthly charge: ${charge_75:.2f}")
print(f"High charge customers: {df['Is_High_Charge'].sum():,} ({df['Is_High_Charge'].mean():.1%})")
print(f"High risk combo (monthly + high charge): {df['High_Risk_Combo'].sum():,} ({df['High_Risk_Combo'].mean():.1%})")
print(f"\nTotal services distribution:\n{df['Total_Services'].value_counts().sort_index()}")

Charge features created.

75th percentile monthly charge: $89.85
High charge customers: 1,758 (25.0%)
High risk combo (monthly + high charge): 870 (12.4%)

Total services distribution:
Total_Services
0      80
1    1701
2    1188
3     965
4     922
5     908
6     676
7     395
8     208
Name: count, dtype: int64


## 5. Demographic features

Senior citizens and customers without partners or dependents
tend to have fewer switching costs and higher churn rates.

In [ ]:
# Senior citizen is already binary (0/1) in the dataset
# Partner and dependents — no support network, easier to churn
df['No_Partner'] = (df['Partner'] == 'No').astype(int)
df['No_Dependents'] = (df['Dependents'] == 'No').astype(int)

# Isolation score — no partner AND no dependents AND senior
df['Isolation_Score'] = (
    df['No_Partner'] +
    df['No_Dependents'] +
    df['SeniorCitizen']
)

print("Demographic features created.")
print(f"\nSenior citizens: {df['SeniorCitizen'].sum():,} ({df['SeniorCitizen'].mean():.1%})")
print(f"No partner: {df['No_Partner'].sum():,} ({df['No_Partner'].mean():.1%})")
print(f"No dependents: {df['No_Dependents'].sum():,} ({df['No_Dependents'].mean():.1%})")
print(f"\nIsolation score distribution:")
print(df['Isolation_Score'].value_counts().sort_index())

Demographic features created.

Senior citizens: 1,142 (16.2%)
No partner: 3,641 (51.7%)
No dependents: 4,933 (70.0%)

Isolation score distribution:
Isolation_Score
0    1666
1    1599
2    3217
3     561
Name: count, dtype: int64


## 6. Verify all engineered features

Quick sanity check — confirm all new features are created correctly
and have no unexpected nulls before building the final feature matrix.

In [ ]:
new_features = [
    'Tenure_Group', 'Is_New_Customer',
    'Is_Monthly_Contract', 'Is_Electronic_Check', 'Is_Paperless',
    'Has_TechSupport', 'Has_OnlineSecurity', 'Has_OnlineBackup', 'Has_DeviceProtection',
    'Protective_Services_Score', 'No_Protective_Services',
    'Total_Services', 'Charge_Per_Service', 'Is_High_Charge', 'High_Risk_Combo',
    'No_Partner', 'No_Dependents', 'Isolation_Score'
]

print(f"Total new features created: {len(new_features)}")
print(f"\nNull check:")
print(df[new_features].isnull().sum())
print(f"\nSample of engineered features:")
df[new_features].head()

Total new features created: 18

Null check:
Tenure_Group                 11
Is_New_Customer               0
Is_Monthly_Contract           0
Is_Electronic_Check           0
Is_Paperless                  0
Has_TechSupport               0
Has_OnlineSecurity            0
Has_OnlineBackup              0
Has_DeviceProtection          0
Protective_Services_Score     0
No_Protective_Services        0
Total_Services                0
Charge_Per_Service            0
Is_High_Charge                0
High_Risk_Combo               0
No_Partner                    0
No_Dependents                 0
Isolation_Score               0
dtype: int64

Sample of engineered features:


,Tenure_Group,Is_New_Customer,Is_Monthly_Contract,Is_Electronic_Check,Is_Paperless,Has_TechSupport,Has_OnlineSecurity,Has_OnlineBackup,Has_DeviceProtection,Protective_Services_Score,No_Protective_Services,Total_Services,Charge_Per_Service,Is_High_Charge,High_Risk_Combo,No_Partner,No_Dependents,Isolation_Score
0,0-12 months,1,1,1,1,0,0,1,0,1,0,1,14.9250,0,0,0,1,1
1,25-48 months,0,0,0,0,0,1,0,1,2,0,3,14.2375,0,0,1,1,2
2,0-12 months,1,1,0,1,0,1,1,0,2,0,3,13.4625,0,0,1,1,2
3,25-48 months,0,0,0,0,1,1,0,1,3,0,3,10.5750,0,0,1,1,2
4,0-12 months,1,1,1,1,0,0,0,0,0,1,1,35.3500,0,0,1,1,2


## 7. Fix Tenure_Group nulls

The 11 customers with tenure = 0 fall outside the bin range.
They are brand new customers — assign them to the 0-12 months group.

In [ ]:
# Fill tenure=0 customers into the 0-12 months bucket
df['Tenure_Group'] = df['Tenure_Group'].cat.add_categories('0-12 months') \
    if '0-12 months' not in df['Tenure_Group'].cat.categories.tolist() \
    else df['Tenure_Group']

df['Tenure_Group'] = df['Tenure_Group'].fillna('0-12 months')

print(f"Nulls remaining in Tenure_Group: {df['Tenure_Group'].isnull().sum()}")
print(f"\nTenure group distribution after fix:")
print(df['Tenure_Group'].value_counts().sort_index())

Nulls remaining in Tenure_Group: 0

Tenure group distribution after fix:
Tenure_Group
0-12 months     2186
13-24 months    1024
25-48 months    1594
49-72 months    2239
Name: count, dtype: int64


## 8. Build the final feature matrix

Select all model-ready features, encode remaining categoricals,
and save the processed dataset for notebook 03.

In [ ]:
# Select final features for modeling
feature_cols = [
    # Tenure
    'tenure', 'Is_New_Customer',
    # Contract and payment
    'Is_Monthly_Contract', 'Is_Electronic_Check', 'Is_Paperless',
    # Services
    'Has_TechSupport', 'Has_OnlineSecurity', 'Has_OnlineBackup',
    'Has_DeviceProtection', 'Protective_Services_Score', 'No_Protective_Services',
    'Total_Services',
    # Charges
    'MonthlyCharges', 'TotalCharges', 'Charge_Per_Service',
    'Is_High_Charge', 'High_Risk_Combo',
    # Demographics
    'SeniorCitizen', 'No_Partner', 'No_Dependents', 'Isolation_Score',
    # Internet service type — encode below
    'InternetService',
    # Gender
    'gender'
]

# Encode remaining categoricals
df_model = df[feature_cols + ['Churn_Binary']].copy()

# Internet service — DSL, Fiber optic, No
df_model = pd.get_dummies(df_model, columns=['InternetService'], drop_first=False)

# Gender — binary encode
df_model['Is_Female'] = (df['gender'] == 'Female').astype(int)
df_model = df_model.drop('gender', axis=1)

print(f"Final feature matrix shape: {df_model.shape}")
print(f"\nFeature columns ({len(df_model.columns)-1} features):")
for col in df_model.columns[:-1]:
    print(f"  {col}")

Final feature matrix shape: (7043, 26)

Feature columns (25 features):
  tenure
  Is_New_Customer
  Is_Monthly_Contract
  Is_Electronic_Check
  Is_Paperless
  Has_TechSupport
  Has_OnlineSecurity
  Has_OnlineBackup
  Has_DeviceProtection
  Protective_Services_Score
  No_Protective_Services
  Total_Services
  MonthlyCharges
  TotalCharges
  Charge_Per_Service
  Is_High_Charge
  High_Risk_Combo
  SeniorCitizen
  No_Partner
  No_Dependents
  Isolation_Score
  Churn_Binary
  InternetService_DSL
  InternetService_Fiber optic
  InternetService_No


## 9. Fix gender encoding and final check

In [ ]:
# Add Is_Female back
df_model['Is_Female'] = (df['gender'] == 'Female').astype(int)

# Final shape check
X = df_model.drop('Churn_Binary', axis=1)
y = df_model['Churn_Binary']

print(f"Feature matrix shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Churn rate in target: {y.mean():.1%}")
print(f"\nFinal feature list ({X.shape[1]} features):")
for col in X.columns:
    print(f"  {col}")

Feature matrix shape: (7043, 25)
Target shape: (7043,)
Churn rate in target: 26.5%

Final feature list (25 features):
  tenure
  Is_New_Customer
  Is_Monthly_Contract
  Is_Electronic_Check
  Is_Paperless
  Has_TechSupport
  Has_OnlineSecurity
  Has_OnlineBackup
  Has_DeviceProtection
  Protective_Services_Score
  No_Protective_Services
  Total_Services
  MonthlyCharges
  TotalCharges
  Charge_Per_Service
  Is_High_Charge
  High_Risk_Combo
  SeniorCitizen
  No_Partner
  No_Dependents
  Isolation_Score
  InternetService_DSL
  InternetService_Fiber optic
  InternetService_No
  Is_Female


## 10. Summary of engineered features

**Total: 25 features ready for modeling**

**Tenure (2 features)**
- `tenure` — raw months with company
- `Is_New_Customer` — flag for first 12 months, highest churn risk window

**Contract and payment (3 features)**
- `Is_Monthly_Contract` — month-to-month churns at 15x two-year rate
- `Is_Electronic_Check` — electronic check customers churn at 45.3%
- `Is_Paperless` — correlated with digital-first, higher churn profile

**Services (7 features)**
- `Has_TechSupport`, `Has_OnlineSecurity`, `Has_OnlineBackup`, `Has_DeviceProtection` — individual protective service flags
- `Protective_Services_Score` — count of protective services (0 to 4)
- `No_Protective_Services` — flag for zero protective services
- `Total_Services` — total number of any services subscribed

**Charges (5 features)**
- `MonthlyCharges`, `TotalCharges` — raw charge amounts
- `Charge_Per_Service` — monthly charge normalized by services used
- `Is_High_Charge` — above 75th percentile ($89.85)
- `High_Risk_Combo` — month-to-month AND high charge, highest risk segment

**Demographics (4 features)**
- `SeniorCitizen`, `No_Partner`, `No_Dependents` — lower switching costs
- `Isolation_Score` — combined isolation signal (0 to 3)

**Internet service (3 features)**
- `InternetService_DSL`, `InternetService_Fiber optic`, `InternetService_No`

**Gender (1 feature)**
- `Is_Female` — control variable

**Next:** Notebook 03 — train logistic regression, random forest,
and XGBoost with 5-fold cross-validation and SHAP analysis.